# 手撕 vLLM-PagedAttention

在上一章节实现了 PageKVCache，解决请求的处理长度产生碎片导致的 batch 上不去的问题。

本文根据 vLLM second meetup 介绍的特性：PageAttention, 右图一个请求可以在多个 block 上算 Attention，通过 FlashAttention 聚合结果，这是核心体现 vLLM 计算高效性：Memory-Efficient。

![kernel](./PageAttention.png)

简要描述上一版本实现的方案：

Attention 的输入是 [num_request, max_pages*pages_size, num_head, head_dim]

PageAttention 的 KVCache 输入是 [num_pages, pages_size, num_head, head_dim]

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from typing import Dict, List, Set, Tuple, Optional

torch.manual_seed(42)

# config

In [4]:
from dataclasses import dataclass

@dataclass
class vLLMEngineConfig:
    max_batch_size = 4
    max_seq_len = 32
    max_prompt_len:int = 16
    
    # model & kv cache
    num_layers = 3

    # PageKV Cache Setting
    block_size = 64
    num_blocks = 1024
    
    dim = 16
    num_heads = 2
    head_dim = 8
    vocab_size = 20

config = vLLMEngineConfig()
print(config.max_seq_len)

32


# PageAttention Kernel

## PageAttention Kernel Prefill

分析 Prefill, 混合 Requests 拼接 block

```
Request 1: Page1, Page7, Page2
Request 2: Page5, Page6
```

此时输入为 `X = [5, page_size, dim]`, 即原本 batchsize 为 2 tensor, 转为 5.

在做 Attention 时可以分别执行

```
Ouput_Request_1 = Attention(X[:3])
Ouput_Request_2 = Attention(X[3:5]) 
```

这里的 Attention 后端实现可以由：

1. 标准 Attention : 将 `X[3, page_size, dim]` 转为 `[1, 3*page_size, dim]`, 其中 1 为batchsize，即为常规的 attention输入
2. Flash Attention: 在 `X[3, page_size, dim]` 基础上天然可以做 Block Attention
3. Flash-Attn: 借用第三方库实现，可以直接做 Diag-Like 的注意力

先实现调用主体函数：根据 Request 做循环

In [9]:
import math
class PageAttentionDecoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_heads = config.num_heads
        self.dim = config.dim
        self.head_dim = config.head_dim
        self.WQ = nn.Linear(config.dim, config.dim, bias=False)
        self.WK = nn.Linear(config.dim, config.dim, bias=False)
        self.WV = nn.Linear(config.dim, config.dim, bias=False)
        self.WO = nn.Linear(config.dim, config.dim, bias=False)
        self.act = nn.ReLU()
        
    def forward_prefill(self,
                        X, 
                        attention_mask=None, # Attention mask 也是 page 型的 
                        request_num_pages=None,
                        request_length=None,
                        attention_backend=None, 
                        ):
        """
        Request-Wise PageAttention Prefill
        Args
            X: input [num_pages, page_size, dim],  num_pages=5.
                here has 2 request, but dim=0 batch_size is not 2
            request_num_pages: list[int],  2 request [3, 2]
            request_length = list[int], 2 request [T1, T2]
            attention_backend: function: MultiHeadsAttention, FlashAttention
        Output
            O:  [num_pages, page_size, dim]
        """
        # Prefill
        B, T, _ = X.shape 
        H = self.num_heads
        D = self.head_dim
        
        Q, K, V = self.WQ(X), self.WK(X), self.WV(X)
        Q=Q.reshape(B, T, H, D).transpose(1,2)
        K=K.reshape(B, T, H, D).transpose(1,2)
        V=V.reshape(B, T, H, D).transpose(1,2)
        O = torch.zeros_like(Q)

        request_size = len(request_num_pages)
        offset = [0] * request_size
        for i in range(1, request_size):
            offset[i] = offset[i-1] + request_num_pages[i]
            
        for t in range(request_size): # Request Loop
            offset_i = offset[t]
            N = request_num_pages[t]
            Q_ = Q[offset_i: offset_i+N]
            K_ = K[offset_i: offset_i+N]
            V_ = V[offset_i: offset_i+N]
    
            O_ = attention_backend(Q_, K_, V_, mask = attention_mask)
            O[offset_i: offset_i+N] = O_
    
        O = O.transpose(1,2).reshape(B, T, H*D)
    
        O = self.WO(O)
        
        O = X + self.act(O)

        return O, [K.transpose(1,2), V.transpose(1,2)]

### PageAttention Kernel: Basic Backend

In [10]:
def MultiHeadsAttention(Q, K, V, mask=None):
    """
    One Request Attention
    Args
        Q: num_pages, num_heads, seq_len, head_dim (in decoding, seq_len=1)
        K: num_pages, num_heads, seq_len, head_dim
        V: num_pages, num_heads, seq_len, head_dim
        mask: num_pages, seq_len, seq_len
    Output
        O: num_pages, num_heads, seq_len, head_dim
    """
    B, H, T, D = Q.shape
    Q = Q.transpose(0,1).reshape(H, B*T, D)
    K = K.transpose(0,1).reshape(H, B*T, D)
    V = V.transpose(0,1).reshape(H, B*T, D)

    S = Q @ K.transpose(1,2)
    P = F.softmax(S, dim = -1)
    Z = P @ V
    O = Z.reshape(H, B, T, D).transpose(0,1)

    return O

In [11]:
def add(a,b):
    return a+b
def div(a,b):
    return a-b
def op(a,b, fun):
    return fun(a,b)
op(5,3, add)

8

In [14]:
model = PageAttentionDecoderBlock(config)
request_num_pages = [3, 2]
page_size = config.block_size
X = torch.randn(5, page_size, config.dim)

O, _ = model.forward_prefill(X, 
                             request_num_pages=[3, 2], 
                             attention_backend=MultiHeadsAttention)
print(O.shape)

torch.Size([5, 64, 16])


### PageAttention Kernel: FlashAttention-V2 Backend

In [16]:
def FlashAttention(Q, K, V, mask=None):
    """
    1 Request(batch_size=1), [Flash Attention-V2](https://zhuanlan.zhihu.com/p/670085985)
    Args
        Q: num_pages, num_heads, seq_len, head_dim (in decoding, seq_len=1)
        K: num_pages, num_heads, seq_len, head_dim
        V: num_pages, num_heads, seq_len, head_dim
        mask: num_pages, seq_len, seq_len
    Output
        O: num_pages, num_heads, seq_len, head_dim
    """
    
    N, H, T, D = Q.shape # batch_size, num_heads, seq_len, head_dim

    O_global = torch.zeros(N, H, T, D)
    for i in range(N): # Q Loop   
        O = torch.zeros(1, H, T, 1)
        M = torch.zeros(1, H, T, 1)
        L = torch.zeros(1, H, T, 1)
        Q_ = Q[i]
        for j in range(N): # KV Loop
            
            if j > i:
                continue
            K_, V_ = K[j], V[j]
            
            S_ij = Q_ @ K_.transpose(1,2) # num_heads, seq_len, seq_len
            M_ij, _ = torch.max(S_ij, dim = -1, keepdim=True) # num_heads, seq_len, 1
            M_new = torch.maximum(M_ij, M)
            P_ij = torch.exp(S_ij - M_new)
            L_ij = torch.sum(P_ij , dim = -1, keepdim=True ) # num_heads, seq_len, 1
            L_new = torch.exp(M - M_new) * L + L_ij
            O_i = torch.exp(M - M_new) * O + P_ij @ V_

            M = M_new
            L = L_new
            
        # re-scaled
        O_global[i] = (O_i / L_new).unsqueeze(dim = 0)
        
    return O_global

In [18]:
O,_ = model.forward_prefill(X, 
                          request_num_pages=[3, 2], 
                          attention_backend=FlashAttention)
print(O.shape)

torch.Size([5, 64, 16])


## PageAttention Kernel Decoding


简易推导, 单条 Request 解码

In [50]:
bsz = 1
num_heads = 1 # ignore
page_size = 5
kv_num_page = 4 # KV 总长度为20, 分成 4 页, 每页大小为 5
d = 16

q = torch.randn(1, 1, d) # bsz, seq_len, dim
K = torch.randn(kv_num_page, page_size, d)
V = torch.randn(kv_num_page, page_size, d)

### standard attention

In [57]:
K_ = K.reshape(1, kv_num_page * page_size, d)
V_ = V.reshape(1, kv_num_page * page_size, d)

S = q @ K_.transpose(1,2)
print(S.shape)
P = F.softmax(S, dim = -1)
O = P @ V_
print(O)
print(O.shape)

torch.Size([1, 1, 20])
tensor([[[-0.0841, -0.1600,  0.4098, -0.4529,  0.3542, -0.1209,  0.0686,
           0.3314, -0.3704, -0.1385, -0.3362, -0.0534,  0.3135,  0.0055,
          -0.6124, -1.0371]]])
torch.Size([1, 1, 16])


### PagedAttention with Online-Softmax Update

In [90]:
# block attention 

S = q @ K.transpose(1,2)
M, _ = torch.max(S, dim = -1, keepdim=True)
S_ = torch.exp(S - M)
L = torch.sum( S_, dim = -1, keepdim=True)
P = (S_/L)
O = P @ V

print(M.shape)
print(S.shape)
print(O.shape)

torch.Size([4, 1, 1])
torch.Size([4, 1, 5])
torch.Size([4, 1, 16])


In [91]:
def online_softmax(x):
    xs = x.split(4)
    M = torch.zeros(4)
    L = torch.zeros(4)
    O = [ torch.zeros(4) for i in range(4)]
    for i, x_ in enumerate(xs):
        m, _ = torch.max(x_, dim = 0)
        s = torch.exp(x_ - m)
        l = s.sum()
        o = s / l
        M[i], L[i], O[i] = m, l, o
    print(M)
    M_max, _ = torch.max(M, dim=0)
    L_new =  torch.sum ( torch.sum(M - M_max) * L )

    for i in range(4):
        O[i] = O[i] * L[i]/L_new * torch.exp( M - M_max)

    return torch.cat(O, dim=0)

x = torch.randn(16)
x_ = online_softmax(x)
print(x_)

tensor([1.5189, 1.5282, 0.0498, 0.8806])
tensor([-0.0044, -0.0207, -0.0129, -0.0065, -0.0021, -0.0149, -0.0026, -0.0295,
        -0.0559, -0.0494, -0.0030, -0.0105, -0.0546, -0.0095, -0.0067, -0.0295])


In [94]:
def combine_result(O, L, M):
    """
    TODO 验证正确性
    """
    M_new, _ = torch.max(M, dim=0, keepdim=True)
    L_new = torch.exp(M - M_new) * L
    L_new = torch.sum(L_new, dim = 0, keepdim=True)
    O_new = torch.exp(M - M_new) * (L/L_new) * O
    return O_new.sum(dim = 0, keepdim=True)

O_fa = combine_result(O, L, M)
print(O_fa)

tensor([[[-0.0841, -0.1600,  0.4098, -0.4529,  0.3542, -0.1209,  0.0686,
           0.3314, -0.3704, -0.1385, -0.3362, -0.0534,  0.3135,  0.0055,
          -0.6124, -1.0371]]])


In [95]:
# 另一种写法：不利用 Flash Attention 也可以实现

S = q @ K.transpose(1,2)
M, _ = torch.max(S, dim = -1, keepdim=True)
M, _ = torch.max(M, dim = 0, keepdim=True) # M global
S_ = torch.exp(S - M) 
L = torch.sum(S_, dim = -1, keepdim=True)
L = torch.sum(L, dim = 0, keepdim = True)
P = (S_/L)
O = P @ V
O_sum = O.sum(dim = 0, keepdim=True)

print(M.shape)
print(S.shape)
print(O_sum)

torch.Size([1, 1, 1])
torch.Size([4, 1, 5])
tensor([[[-0.0841, -0.1600,  0.4098, -0.4529,  0.3542, -0.1209,  0.0686,
           0.3314, -0.3704, -0.1385, -0.3362, -0.0534,  0.3135,  0.0055,
          -0.6124, -1.0371]]])


### Kernel 实现

解码任务拟定

Request 有 4 个, KVBlcok pages_num `[2,1,3,2]` 

In [24]:
q = torch.tensor([1,2,3,4])
num_pages = torch.tensor([2,1,3,2])
torch.repeat_interleave(q, num_pages, dim = 0)

tensor([1, 1, 2, 3, 3, 3, 4, 4])

In [22]:
import math
class PageAttentionDecoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_heads = config.num_heads
        self.dim = config.dim
        self.head_dim = config.head_dim
        self.WQ = nn.Linear(config.dim, config.dim, bias=False)
        self.WK = nn.Linear(config.dim, config.dim, bias=False)
        self.WV = nn.Linear(config.dim, config.dim, bias=False)
        self.WO = nn.Linear(config.dim, config.dim, bias=False)
        self.act = nn.ReLU()
        
    def forward_decoding(self,
                        X, 
                        attention_mask=None, # Attention mask 也是 page 型的 
                        request_num_pages=None,
                        request_length=None,
                        KV_Cache=None
                        ):
        """
        Request-Wise PageAttention Decoding
        Args
            X: input [num_pages, page_size, dim],  num_pages=5.
                here has 2 request, but dim=0 batch_size is not 2
            request_num_pages: list[int],  2 request [3, 2]
            request_length = list[int], 2 request [T1, T2]
            KV_Cache = [num_pages, page_size, num_head, head_dim]
        Output
            O:  [num_pages, page_size, dim]
        """
        # Decoding
        B, T, _ = X.shape 
        H = self.num_heads
        D = self.head_dim
        
        # Proj
        q, k, v = self.WQ(X), self.WK(X), self.WV(X)
        q = q.reshape(B, 1, H, D).transpose(1,2)
        k = k.reshape(B, 1, H, D).transpose(1,2)
        v = v.reshape(B, 1, H, D).transpose(1,2)

        # TODO: Apply RoPE For Q,K

        # step1: init
        S = q @ k.transpose(2,3) # B, H, 1, 1)
        M_ = S.clone()
        L_ = torch.ones_like(M_)
        O_ = v

        # step2: repeat q (dispatch)
        repeat_tensor = torch.tensor(request_num_pages)
        q_ = torch.repeat_interleave(q, repeat_tensor, dim = 0) 

        # step3: block attenion, 
        K_, V_ = KV_Cache[0], KV_Cache[1] # bsz, seq_len, num_head, head_dim
        S = q_ @ K_.transpose(1,2).transpose(2,3)
        # TODO: Mask 
        M, _ = torch.max(S, dim=-1, keepdim=True)
        L = torch.sum( torch.exp( S - M), dim=-1, keepdim=True)
        P = torch.softmax(S, dim=-1)
        O = P @ V_.transpose(1,2)

        # step4: reudce result, (combine)
        offset = 0
        globle_O = torch.zeros_like(O_)
        for i, T in enumerate(request_num_pages):
            Oi = self.combine_result(
                O[offset:offset+T],
                M[offset:offset+T],
                L[offset:offset+T],
                O_[i],
                M_[i],
                L_[i],
            )
            globle_O[i] = Oi[0] # BH1D
            offset += T
            break
            
        O = globle_O.transpose(1,2).reshape(B, 1, H*D)
        O = self.WO(O)
        O = X + self.act(O)
        
        return O, [k.transpose(1,2), v.transpose(1,2)]

    def combine_result(self, O, M, L, O_, M_, L_):
        """
        online softmax trick
        """
        O = torch.cat( [O, O_.unsqueeze(dim=0)], dim = 0)
        M = torch.cat( [M, M_.unsqueeze(dim=0)], dim = 0)
        L = torch.cat( [L, L_.unsqueeze(dim=0)], dim = 0)

        M_new,_ = torch.max(M, dim=0, keepdim=True)
        L_new =  torch.exp(M-M_new) * L
        O_new = (M-M_new) * (L/L_new) * O 
        
        O_new = torch.sum(O_new, keepdim=True, dim=0)
        
        return O_new

In [23]:
model = PageAttentionDecoderBlock(config)
request_num_pages = [3, 1, 2, 2]
page_size = config.block_size
X = torch.randn(4, 1, config.dim) # 4 个请求
k_cache = torch.randn(8, page_size, config.num_heads, config.head_dim)
v_cache = torch.randn(8, page_size, config.num_heads, config.head_dim)

O, tmp_kv_cache = model.forward_decoding(X, request_num_pages=[3, 1, 2, 2], KV_Cache= [k_cache, v_cache])
print(X.shape)
print(O.shape)

torch.Size([4, 1, 16])
torch.Size([4, 1, 16])
